In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
%sql
SELECT * from bronze_reviews
LIMIT 100

In [0]:
%sql
CREATE OR REPLACE TABLE silver_reviews AS
WITH parsed AS (
  SELECT
    game_name,
    recommendation,
    date,

    CAST(NULLIF(REGEXP_EXTRACT(review, '^(\\d{4})\\s'), '') AS INT) AS review_year,
    TRIM(REGEXP_REPLACE(review, '^\\d{4}\\s', '')) AS review_text,

    TRY_CAST(REPLACE(hours_played, ',', '') AS DOUBLE) AS hours_played,
    TRY_CAST(REPLACE(helpful, ',', '') AS INT) AS helpful,
    TRY_CAST(REPLACE(funny, ',', '') AS INT) AS funny,

    COALESCE(
      TRY_TO_DATE(date, 'MMMM d, yyyy'),
      TRY_TO_DATE(date, 'd MMMM, yyyy'),
      TRY_TO_DATE(CONCAT(date, ', ', CAST(NULLIF(REGEXP_EXTRACT(review, '^(\\d{4})\\s'), '') AS INT)), 'MMMM d, yyyy'),
      TRY_TO_DATE(CONCAT(date, ', ', CAST(NULLIF(REGEXP_EXTRACT(review, '^(\\d{4})\\s'), '') AS INT)), 'd MMMM, yyyy')
    ) AS review_date,

    NULLIF(TRIM(REGEXP_EXTRACT(username, '^([^\\n]+)')), '') AS username,
    CAST(NULLIF(REGEXP_REPLACE(REGEXP_EXTRACT(username, '([\\d,]+) products'), ',', ''), '') AS INT) AS reviewer_products

  FROM bronze_reviews
  WHERE review IS NOT NULL AND game_name IS NOT NULL
)

SELECT DISTINCT
  game_name,
  recommendation,
  review_text,
  hours_played,
  helpful,
  funny,
  username,
  reviewer_products,
  date            AS date_raw,
  review_year,
  review_date,
  CASE WHEN review_date IS NOT NULL THEN 1 ELSE 0 END AS has_date
FROM parsed
WHERE TRIM(review_text) <> ''

-- date column comes with two shapes
  -- With a year: "November 27, 2019", "7 December, 2023"
  -- Without a year: "March 21", "21 May"
-- We noticed that review (the actual review text column) often has a 4-digit year glued onto the very front of it — e.g. review = "2019 This game is amazing..."
  -- we scrape that 2019 into its own column: review_year
-- Now we try to build the real date column, review_date
  -- try to parse date
  -- if date has no year: glue review_year onto date and try "Month D, Year" again.

In [0]:
%sql
SELECT * FROM silver_reviews
WHERE review_date IS NOT NULL
LIMIT 25